# Lab 6.1: Tensor Parallelism

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/07_scaling/06.1_tensor_parallelism/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/open?url=https://github.com/harshuljain13/llm-inference-at-scale/blob/master/content/07_scaling/06.1_tensor_parallelism/lab.ipynb)

Simulate column-parallel and row-parallel splits, measure AllReduce cost,
and calculate scaling efficiency across interconnects.

In [ ]:
# Install dependencies via subprocess (Colab/Molab compatible)
import subprocess, sys
# Use pip to install numpy and matplotlib in the current kernel
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
# numpy: used for matrix multiplication simulations
import numpy as np
# matplotlib: used for plotting scaling efficiency charts
import matplotlib.pyplot as plt

# Set random seed so all weight matrices are reproducible
np.random.seed(42)

## Exercise 1: Column-Parallel Linear Layer

Split weight matrix W along columns. Each GPU computes a disjoint slice of the output.

In [ ]:
def exercise_1_column_parallel():
    """Demonstrate column-parallel split across GPUs."""

    def column_parallel_linear(X, W, num_gpus):
        """Split W along columns: each GPU gets W[:, shard_start:shard_end]."""
        # Ensure columns divide evenly across GPUs
        assert W.shape[1] % num_gpus == 0
        # Number of columns each GPU is responsible for
        shard_size = W.shape[1] // num_gpus
        # Collect output from each simulated GPU
        outputs = []
        for i in range(num_gpus):
            # Slice: this GPU's portion of the weight matrix
            W_shard = W[:, i * shard_size:(i + 1) * shard_size]
            # Matmul: full input X times column slice -> partial output
            outputs.append(X @ W_shard)
        # Return list of partial outputs (no communication needed yet)
        return outputs

    # --- Configuration ---
    # Flattened batch*seq dimension (e.g., batch=2, seq=128)
    batch_seq = 256
    # Hidden dimension of the transformer model
    hidden = 4096
    # FFN intermediate dimension (typically 4x hidden)
    ffn_dim = 16384
    # Number of GPUs for tensor parallelism
    num_gpus = 4

    # Create random input activation tensor
    X = np.random.randn(batch_seq, hidden).astype(np.float32)
    # Create random weight matrix (first FFN linear)
    W_up = np.random.randn(hidden, ffn_dim).astype(np.float32) * 0.01

    # Execute column-parallel split across 4 simulated GPUs
    shards = column_parallel_linear(X, W_up, num_gpus)

    # Verify correctness: concatenating shards = full single-GPU matmul
    full_output = X @ W_up
    # Concatenate along column axis to reconstruct full result
    reconstructed = np.concatenate(shards, axis=1)
    # Numerical error should be near zero (floating point only)
    max_error = np.max(np.abs(full_output - reconstructed))

    # Print shapes and verification
    print(f'Input shape: {X.shape}')
    print(f'Weight shape: {W_up.shape}')
    print(f'Each GPU output shard: {shards[0].shape}')
    print(f'Reconstruction error: {max_error:.2e} (should be ~0)')
    # Return shards and weights for use in Exercise 2
    return shards, W_up, X, full_output

# Run exercise 1
shards_ex1, W_up_ex1, X_ex1, full_out_ex1 = exercise_1_column_parallel()

## Exercise 2: Row-Parallel Linear + AllReduce

Split W along rows. Each GPU computes a partial sum. AllReduce sums them.

In [ ]:
def exercise_2_row_parallel(input_shards, full_output_prev):
    """Demonstrate row-parallel split with AllReduce."""

    def row_parallel_linear(input_shards, W, num_gpus):
        """Split W along rows. Sum partial outputs = AllReduce."""
        # Verify rows divide evenly
        assert W.shape[0] % num_gpus == 0
        # Number of rows per GPU
        shard_size = W.shape[0] // num_gpus
        # Accumulate partial sums from each GPU
        partial_sums = []
        for i in range(num_gpus):
            # This GPU's row slice of the weight matrix
            W_shard = W[i * shard_size:(i + 1) * shard_size, :]
            # Multiply input shard (from column-parallel) by row slice
            partial_sums.append(input_shards[i] @ W_shard)
        # AllReduce operation: element-wise sum of all partial results
        # In real systems, NCCL performs this across GPU memory
        result = sum(partial_sums)
        return result

    # Number of GPUs matches column-parallel split
    num_gpus = len(input_shards)
    # FFN dimension inferred from input shard width
    ffn_dim = input_shards[0].shape[1] * num_gpus
    # Hidden dimension = output width of row-parallel layer
    hidden = 4096
    # Create second FFN linear weight (projects back to hidden)
    W_down = np.random.randn(ffn_dim, hidden).astype(np.float32) * 0.01

    # Execute row-parallel with AllReduce
    result = row_parallel_linear(input_shards, W_down, num_gpus)

    # Verify: should match sequential two-matmul computation
    reference = full_output_prev @ W_down
    # Error from floating-point accumulation order differences
    allreduce_error = np.max(np.abs(result - reference))

    print(f'Row-parallel output shape: {result.shape}')
    print(f'AllReduce verification error: {allreduce_error:.2e}')
    print(f'Communication: 1 AllReduce per FFN block ({num_gpus} GPUs)')
    return result

# Run exercise 2 using outputs from exercise 1
_ = exercise_2_row_parallel(shards_ex1, full_out_ex1)

## Exercise 3: AllReduce Communication Cost

Ring AllReduce transfers 2*(N-1)/N * message_size bytes. Compare interconnects.

In [ ]:
def exercise_3_allreduce_cost():
    """Model AllReduce communication overhead for Llama 70B."""

    def allreduce_time_ms(msg_bytes, num_gpus, bw_gb_per_s):
        """Ring AllReduce latency in milliseconds."""
        # Ring algorithm data volume formula
        volume_bytes = 2 * (num_gpus - 1) / num_gpus * msg_bytes
        # Convert GB/s to bytes/ms for consistent units
        bw_bytes_per_ms = bw_gb_per_s * 1e9 / 1e3
        # Latency = data volume / bandwidth
        return volume_bytes / bw_bytes_per_ms

    # --- Llama 70B architecture parameters ---
    # Hidden dimension determines message size
    hidden_70b = 8192
    # Number of transformer layers (each needs 2 AllReduces)
    layers_70b = 80
    # Batch size: concurrent sequences during decode
    batch_size = 32
    # Tensor parallel degree
    tp_degree = 8

    # Message size per AllReduce: batch * hidden * 2 bytes (FP16)
    msg_bytes = batch_size * hidden_70b * 2
    # Total AllReduces per forward pass: 2 per layer (attn + FFN)
    total_allreduces = 2 * layers_70b

    # Interconnect bandwidth comparison
    interconnects = {
        'NVLink 4 (H100)': 900,   # GB/s bidirectional
        'NVLink 3 (A100)': 600,   # GB/s bidirectional
        'PCIe 5.0': 64,           # GB/s
        'PCIe 4.0': 32,           # GB/s
    }

    # Print comparison table
    print(f'Llama 70B decode: TP={tp_degree}, batch={batch_size}')
    print(f'Message size per AllReduce: {msg_bytes/1024:.1f} KB')
    print(f'AllReduces per forward: {total_allreduces}')
    print()
    print(f'{"Interconnect":<22}{"Per-AR (us)":<14}{"Total/fwd (ms)":<16}{"% of 10ms budget"}')
    print('-' * 66)
    for name, bw in interconnects.items():
        # Time for single AllReduce
        t_single = allreduce_time_ms(msg_bytes, tp_degree, bw)
        # Total communication per forward pass
        t_total = t_single * total_allreduces
        # Percentage of a 10ms decode budget consumed by communication
        pct = t_total / 10 * 100
        print(f'{name:<22}{t_single*1000:<14.2f}{t_total:<16.3f}{pct:.1f}%')

# Run exercise 3
exercise_3_allreduce_cost()

## Exercise 4: Scaling Efficiency Chart

Plot efficiency = compute / (compute + communication) vs TP degree.

In [ ]:
def exercise_4_efficiency_chart():
    """Plot TP scaling efficiency for different interconnects."""

    def tp_efficiency(params_b, hidden, layers, batch, num_gpus, bw_gb_s):
        """Compute TP efficiency: useful_compute / total_time."""
        # Forward pass FLOPs: ~2 * params per token
        flops_per_fwd = 2 * params_b * 1e9 * batch
        # A100 FP16 peak throughput
        gpu_peak_flops = 312e12
        # Compute time split across GPUs (ms)
        compute_ms = flops_per_fwd / (gpu_peak_flops * num_gpus) * 1e3
        # Communication: 2 AllReduces per layer
        msg_bytes = batch * hidden * 2  # FP16 hidden state
        # Ring AllReduce volume
        ar_volume = 2 * (num_gpus - 1) / num_gpus * msg_bytes
        # Time per AllReduce in ms
        ar_time_ms = ar_volume / (bw_gb_s * 1e9 / 1e3)
        # Total communication across all layers
        comm_ms = ar_time_ms * 2 * layers
        # Efficiency: compute fraction of total time
        if num_gpus == 1:
            return 1.0
        return compute_ms / (compute_ms + comm_ms)

    # TP degrees to evaluate
    tp_degrees = [1, 2, 4, 8]
    # Interconnect configurations to compare
    configs = {
        'NVLink 4 (900 GB/s)': 900,
        'NVLink 3 (600 GB/s)': 600,
        'PCIe 5.0 (64 GB/s)': 64,
    }

    # Create figure with single subplot
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    # Plot efficiency curve for each interconnect
    for label, bw in configs.items():
        # Calculate efficiency at each TP degree (Llama 70B, batch=32)
        effs = [tp_efficiency(70, 8192, 80, 32, tp, bw) * 100 for tp in tp_degrees]
        # Plot with markers for visibility
        ax.plot(tp_degrees, effs, 'o-', label=label, linewidth=2, markersize=8)

    # Ideal 100% efficiency reference line
    ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Ideal')

    # Axis labels and formatting
    ax.set_xlabel('Tensor Parallel Degree', fontsize=12)
    ax.set_ylabel('Scaling Efficiency (%)', fontsize=12)
    ax.set_title('Llama 70B: TP Efficiency by Interconnect (batch=32)', fontsize=13)
    # Only show valid TP degrees on x-axis
    ax.set_xticks(tp_degrees)
    # Y range shows the full degradation story
    ax.set_ylim(40, 105)
    ax.legend(fontsize=10)
    # Light grid for readability
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Run exercise 4
exercise_4_efficiency_chart()

## Exercise 5: Per-GPU VRAM Calculator

Calculate memory breakdown: weights/TP + KV cache + overhead.

In [ ]:
def exercise_5_vram_calculator():
    """Calculate per-GPU VRAM for different models and TP degrees."""

    def vram_per_gpu_gb(params_b, num_layers, num_heads, head_dim,
                        num_gpus, batch, seq_len, dtype_bytes=2):
        """Per-GPU memory: weights + KV cache + overhead."""
        # Model weights divided evenly across GPUs
        weights_gb = (params_b * 1e9 * dtype_bytes) / (num_gpus * 1024**3)
        # Attention heads assigned to this GPU
        heads_per_gpu = num_heads // num_gpus
        # KV cache: 2 (K+V) * layers * batch * seq * heads * dim * precision
        kv_gb = (2 * num_layers * batch * seq_len * heads_per_gpu * head_dim * dtype_bytes) / 1024**3
        # Framework overhead: NCCL buffers, activations (~20% of weight memory)
        overhead_gb = weights_gb * 0.2
        # Sum all components for total per-GPU requirement
        total_gb = weights_gb + kv_gb + overhead_gb
        return {'weights': weights_gb, 'kv_cache': kv_gb,
                'overhead': overhead_gb, 'total': total_gb}

    # Models to evaluate: (name, params_B, layers, Q_heads, head_dim)
    models = [
        ('Llama 8B', 8, 32, 32, 128),
        ('Llama 70B', 70, 80, 64, 128),
        ('Llama 405B', 405, 126, 128, 128),
    ]

    # Print formatted table header
    print(f'{"Model":<12}{"TP":<5}{"Weights":<10}{"KV Cache":<10}{"Total":<8}{"Fits 80GB?"}')
    print('-' * 55)

    for name, params, layers, heads, hdim in models:
        # Try each TP degree
        for tp in [1, 2, 4, 8]:
            # Skip invalid configurations (heads must divide evenly)
            if heads % tp != 0:
                continue
            # Calculate with batch=8, context=4096
            v = vram_per_gpu_gb(params, layers, heads, hdim, tp, batch=8, seq_len=4096)
            # Check against A100 80GB capacity
            fits = '  Yes' if v['total'] < 80 else '  No'
            print(f'{name:<12}{tp:<5}{v["weights"]:<10.1f}{v["kv_cache"]:<10.1f}'
                  f'{v["total"]:<8.1f}{fits}')

# Run exercise 5
exercise_5_vram_calculator()

## Summary

- **Column-parallel**: splits output dimension, no communication needed
- **Row-parallel**: splits input dimension, requires one AllReduce (sum)
- **Attention heads**: distributed evenly, communication only at output projection
- **NVLink >> PCIe**: 10-28x bandwidth gap makes NVLink mandatory for TP > 2
- **VRAM scales as 1/TP** for weights; KV cache scales by heads/TP